## Import neccessary Python packages

In [ ]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox
from scripts.onsset import *
from pathlib import Path
import sys

## Import the csv file with extracted GIS data to be calibrated

In [ ]:
root_dir = Path.cwd()
sys.path.insert(0, str(root_dir))
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
messagebox.showinfo('OnSSET', 'Open the input file with extracted GIS data')
input_file = filedialog.askopenfilename(initialdir=os.path.join(root_dir, 'inputs', 'extracted_csv'))
#input_file = r"inputs\extracted_csv\InputFile.csv"
onsseter = SettlementProcessor(input_file)

onsseter.conditioning()

## Enter key demographic and electrification parameters

In [ ]:
provinces = ['GAZA', 'TETE', 'MAPUTO', 'MANICA', 'INHAMBANE', 'SOFALA', 'NIASSA', 'ZAMBEZIA', 'NAMPULA', 'CABO DELGADO']

### Calibrate population and urban/rural status

In [ ]:
start_year = 2024
pop_start_year = 33244414       ### Write the population in the base year (e.g. 2024)
urban_ratio_start_year = 0.3486 ### Write the urban population population ratio in the base year (e.g. 2024)
num_people_per_hh_urban = 4.7     ### Write the number of people per household in urban areas
num_people_per_hh_rural = 4.5   ### Write the number of people per household  in rural areas

In [ ]:
pop_modelled, urban_modelled = onsseter.calibrate_current_pop_and_urban(pop_start_year, urban_ratio_start_year)

### Define the household size in each settlement based on the province household size

In [ ]:
df = onsseter.df

In [ ]:
# Here define the number of people per household in each province 
hh_size = {
    'CABO DELGADO': 4.5,
    'ZAMBEZIA': 4.6,
    'SOFALA': 4.9,
    'INHAMBANE': 4.1, 
    'TETE': 4.5,
    'NAMPULA': 4.6,
    'NIASSA': 4.9,
    'GAZA': 4.6,
    'MANICA': 4.9,
    'MAPUTO': 4.3,
}

In [ ]:
df['NumPeoplePerHH'] = 4.6  # National average number of people per household in each province

for p in provinces:
    df.loc[df.Admin_1 == p, 'NumPeoplePerHH'] = hh_size[p]  

In [ ]:
df['FinalElecCode{}'.format(start_year)] = 99
df['ElecPop{}'.format(start_year)] = 0
df['ElecStart'] = 0
df['Pop{}'.format(start_year)] = df['PopStartYear']
df['HHs{}'.format(start_year)] = df['Pop{}'.format(start_year)] / df['NumPeoplePerHH']

### Identify mini-grid electrified settlements

In [ ]:
df.loc[df['id'].isin([518021, 25447, 6408]), 'MGDist'] = 0

In [ ]:
# All settlements where there is a mini-grid (MGDist == 0) are considered mini-grid electrified 
df.loc[df['MGDist'] == 0, 'ElecStart'] = 1
df.loc[df['MGDist'] == 0, 'FinalElecCode{}'.format(start_year)] = 5
df.loc[df['MGDist'] == 0, 'ElecPop{}'.format(start_year)] = df['Pop{}'.format(start_year)]

In [ ]:
mg_ids = [13, 943, 2016, 2197, 2459, 6408, 11496, 11507, 13950, 19578, 19587, 19607, 20213, 24704, 24720, 24726, 25447, 25577, 30707, 34577, 37333, 
          40425, 40431, 46561, 50798, 57396, 64434, 65384, 70300, 73338, 78161, 85058, 110114, 157063, 177112, 180179, 185665, 193190, 195320, 209924, 
          211617, 219705, 219707, 281745, 284974, 294765, 298477, 302178, 304590, 306201, 319505, 345743, 388043, 401429, 434880, 498355, 500584, 
          508441, 508450, 511657, 514434, 518021, 518601, 539638, 574036, 582399, 591630, 598383, 604944, 644876, 670858, 732635, 752455, 784998, 
          819115, 822354, 859374, 863203, 863277, 870309, 876718, 886614, 893211, 909570, 909576, 909579, 912389, 912889, 939052, 958133, 971274, 
          984950, 988051, 993748, 1043038, 1059381, 1071818]

mg_elec_pop = [460.0, 229.0535, 300.3746, 1184.6433, 768.5416, 1435.5, 85.7123, 27.2609, 47.3724, 148.5, 148.5, 153.0, 131.0426, 30.8641, 36.5189,
               5.7365, 61.8677, 474.9975, 225.0595, 334.4156, 483.3838, 19.3675, 14.5082, 350.3957, 14.8013, 18.6234, 237.9873, 41.9441, 253.8686, 
               6.6416, 386.7687, 19.0701, 96.4412, 65.0843, 424.2968, 357.0684, 144.2195, 331.462, 28.9616, 35.178, 377.6201, 8.8468, 44.5294, 963.0195, 
               200.1789, 866.2315, 239.2198, 49.7258, 446.1691, 646.2965, 380.0691, 525.1453, 588.0, 563.5, 628.6457, 40.8543, 55.1198, 795.5331, 
               834.0157, 1196.0, 7.9482, 276.0, 491.4501,  533.5767, 614.5971, 506.0076, 668.5048, 377.6663, 244.1307, 494.1857, 577.842, 739.8957, 
               189.7141, 717.0464, 530.0, 530.0, 800.4, 1296.0984, 1105.5554, 5014.0, 175.5012, 491.9772, 666.4, 1840.5, 458.1349, 482.6054, 920.0, 
               2336.0316, 172.4681, 1243.8935, 508.8571, 598.0, 1800.0, 694.6, 2831.9319, 1472.0, 920.0]

i = 0

for i in range(len(mg_ids)):
    # All settlements where there is a mini-grid (MGDist == 0) are considered mini-grid electrified 
    df.loc[df['id'] == mg_ids[i], 'ElecStart'] = 1
    df.loc[df['id'] == mg_ids[i], 'FinalElecCode{}'.format(start_year)] = 5
    df.loc[df['id'] == mg_ids[i], 'ElecPop{}'.format(start_year)] = mg_elec_pop[i]
    i += 1

### Identify grid-electrified settlements

#### First, identify settlements which are directly on the MV line, and has a minimum number of population

In [ ]:
min_pop = 200

df.loc[((df['MV_overlap'] > 0) | (df['CurrentMVLineDist'] == 0)) & (df['Pop{}'.format(start_year)] > min_pop), 'ElecStart'] = 1
df.loc[((df['MV_overlap'] > 0) | (df['CurrentMVLineDist'] == 0)) & (df['Pop{}'.format(start_year)] > min_pop), 'FinalElecCode{}'.format(start_year)] = 1

#### Next, identify settlements close to the MV lines that display night-time lights that are also likely electrified

In [ ]:
max_mv_line_distance = 3  # Distance  in km from the existing grid network below which we can assume a settlement could be electrified
min_pop = 500      ### Settlement population above which we can assume that it could be electrified

df.loc[(df['CurrentMVLineDist'] < max_mv_line_distance) & (df['Pop{}'.format(start_year)] > min_pop) & (df['NightLights'] > 0), 'ElecStart'] = 1
df.loc[(df['CurrentMVLineDist'] < max_mv_line_distance) & (df['Pop{}'.format(start_year)] > min_pop) & (df['NightLights'] > 0), 'FinalElecCode{}'.format(start_year)] = 1

#### Next, identify settlements that are within 500m from a distribution transformer

In [ ]:
df.loc[df['TRxDistNew'] < 0.5, 'ElecStart'] = 1
df.loc[df['TRxDistNew'] < 0.5, 'FinalElecCode{}'.format(start_year)] = 1

#### Next, identify settlements that have been manually confirmed as electrified

In [ ]:
df.loc[df['id'].isin([408092, 66952, 177066, 21994, 4760, 15992, 581595, 198803, 188941, 211767]), 'ElecStart'] = 1
df.loc[df['id'].isin([408092, 66952, 177066, 21994, 4760, 15992, 581595, 198803, 188941, 211767]), 'FinalElecCode{}'.format(start_year)] = 1

#### Finally, calibrate against the number of EDM connected households per province

In [ ]:
# The number of households connected to EDM per province in the start year of the analysis (e.g. 2024)
elec_hhs = {
    'CABO DELGADO': 174035,
    'ZAMBEZIA':  324673,
    'SOFALA':  341560,
    'INHAMBANE':  148373, 
    'TETE':  202169,
    'NAMPULA':  611074,
    'NIASSA':  243761,
    'GAZA':  254848,
    'MANICA':  199753,
    'MAPUTO':  852289,
}

In [ ]:
for p in provinces:
    elec_area_pop = df.loc[(df.Admin_1 == p) & (df['FinalElecCode{}'.format(start_year)] == 1), 'Pop{}'.format(start_year)].sum()
    ratio = min(elec_hhs[p] / (elec_area_pop / hh_size[p]), 1)
    df.loc[(df.Admin_1 == p) & (df['FinalElecCode{}'.format(start_year)] == 1), 'ElecPop{}'.format(start_year)] = df['Pop{}'.format(start_year)] * ratio

### Save as csv

In [ ]:
output_file = r'outputs\calibration\Calibrated.csv'

df['ElecPopCalib'] = df['ElecPop{}'.format(start_year)]
df.to_csv(output_file, index=False)